In [2]:
import jax 
import jax.numpy as jnp 

from functools import partial

jax.config.update("jax_enable_x64", True)

# Adjoints

We have the following situation with an objective
$$f(\theta,x)$$
and a PDE constrained $x$:

$$A(\theta)x = b(\theta)$$

or equivalently:

$$r(\theta,x) = A(\theta)x -b(\theta) = 0$$


Furthermore, we define 

$$A^T \psi = \left[\frac{\partial f}{\partial x}\right]^T$$

(the transpose is because multiple f values mean multiple solves). Then, we have 
$$\frac{df}{d\theta} = \frac{\partial f}{\partial \theta } - \psi^T \frac{\partial r}{\partial \theta}$$
This has two different components:
1. The direct dependence of f on $\theta$
2. The indirect dependence of $f$ on $\theta$ through the state equation. This is a Vector-Jacobian Product (VJP) of the adjoint with the residual Jacobian.


Note that JAX has implemented the linear solve JVP itself; so actually there is no need to do this in simple cases, however, we will later use a operator definition of $A(\theta)$, and especially $A^{-1}(\theta)$, which we *cannot* differentiate through.
This is implemented in the below example:

In [3]:

def f(x, theta):
    return jnp.sum(x ** 2) + theta[0] * jnp.sum(x) +x[0]*theta[1] # arbitrary objective

def A_param(theta):
    return jnp.array([[1.0, theta[0]],[theta[1], 1.0]])

def b_param(theta):
    return jnp.array([1.0 - 0.1**theta[1], 1.0 - 0.2**theta[0]])

def solve(A,b):
    return jnp.linalg.solve(A,b)

def solve_transpose(A,b):
    return jnp.linalg.solve(A.T,b)


def residual(x, theta):
    return A_param(theta) @ x - b_param(theta)
    
def df_dtheta(theta):
    A = A_param(theta)
    b = b_param(theta)
    x = solve(A,b)
    pf_ptheta = jax.grad(f, argnums=1)(x, theta)
    pf_px     = jax.grad(f, argnums=0)(x, theta)
    psi = solve_transpose(A, pf_px)    
    _, vj_thing = jax.vjp(partial(residual,x),theta)    
    vjp_=  vj_thing( - psi)
    return pf_ptheta + vjp_[0]

theta_base = jnp.array([0.5, 0.3])

f_t = lambda theta: f(solve(A_param(theta), b_param(theta)), theta)
print("f:                            ", f_t(theta_base))
print("df_dtheta using adjoint:      ", df_dtheta(theta_base))
print("df_dtheta using JAX gradient: ", jax.grad(f_t)(theta_base))


f:                             0.7398966031191584
df_dtheta using adjoint:       [0.906369   1.22608691]
df_dtheta using JAX gradient:  [0.906369   1.22608691]


Now, in the future, we want to be able to compose the function:

$G(\theta) = g(\theta) \circ f(\theta)$

and be able to do 
$\frac{dG}{d\theta}$ 
without writing a custom adjoint again.


[Again there is no need to do this for this simple example; JAX already has a custom VJP and JVP implemented for a jnp.linalg.solve]

In [4]:
@partial(jax.custom_vjp)
def F(theta):
    return f(solve(A_param(theta), b_param(theta)), theta)


def F_fwd(theta):
    A = A_param(theta)
    b = b_param(theta)
    x = solve(A,b)
    y = f(x, theta)
    return y, (x, theta)

def F_bwd(res, g):
    x, theta = res
    pf_ptheta = jax.grad(f, argnums=1)(x, theta)
    pf_px     = jax.grad(f, argnums=0)(x, theta)
    A = A_param(theta)

    psi = solve_transpose(A, pf_px)    
    _, vj_f = jax.vjp(partial(residual,x),theta)    

    vjp_=  vj_f( - psi)
    return (g * (pf_ptheta + vjp_[0]),)

def g(f, theta):
    return theta[0] ** f + theta[1]

F.defvjp(F_fwd, F_bwd)

print("JAX direct gradient")
G_theta_jax = lambda t : g(f(solve(A_param(t), b_param(t)), t), t)

print("\tG:        ",    G_theta_jax(theta_base))
print("\tdG_dtheta :",    jax.grad(G_theta_jax)(theta_base))

print("JAX custom VJP gradient")
G_theta_custom_vjp = lambda t : g(F(t), t)
print("\tG:        ",   G_theta_custom_vjp(theta_base))
print("\tdG_dtheta :",    jax.grad(G_theta_custom_vjp)(theta_base))


JAX direct gradient
	G:         0.8987822650513859
	dG_dtheta : [0.5098907  0.49111969]
JAX custom VJP gradient
	G:         0.8987822650513859
	dG_dtheta : [0.5098907  0.49111969]


Actually, JAX implements this machinery directly through jax.lax.custom_linear_solve and this can be used as well (and is in fact internally for jnp.linalg.solve!)

Here we make a little non differentiable solve that errors out whenever one tries to use AD on it.

Then we use the jax.lax.custom_linear_solve on it, this obtains the same result as our previous method. Note that it does *not* differentiate through the solver; that would have produced an error

In [5]:
@jax.custom_vjp
def _non_differentiable_solve(A, b, transpose = False):

        
    luff = jax.scipy.linalg.lu_factor(jax.lax.stop_gradient(A))
    
    return jax.scipy.linalg.lu_solve(luff, b)

def _non_differentiable_solve_fwd(A, b, transpose):
    x = _non_differentiable_solve(A, b, transpose)
    return x, (A, x, transpose)
def _non_differentiable_solve_bwd(res, g):  
    raise NotImplementedError("This solve is non-differentiable")

_non_differentiable_solve.defvjp(_non_differentiable_solve_fwd, _non_differentiable_solve_bwd)


def do_custom_linear_solve(theta):
    A = A_param(theta)
    b = b_param(theta)
    return jax.lax.custom_linear_solve(matvec = lambda x : A @ x, b=b, solve = lambda _, x : _non_differentiable_solve(A,x), transpose_solve = lambda _, x : _non_differentiable_solve(A.T, x))    

print("Custom non differentiable solve:")
try:
    g_non_diff = lambda t : g(f(_non_differentiable_solve(A_param(t), b_param(t)), t), t)
    print("\tG          \t:",    g_non_diff(theta_base))
    
    print("\tdG_dtheta  \t:",     jax.grad(g_non_diff)(theta_base))
except NotImplementedError as e:
    
    print("\tdG_dtheta  \t:","NotImplementedError")
    print("\t\tERROR:", e)

print("Custom linear solve")
g_diff = lambda t : g(f(do_custom_linear_solve(t), t), t)
print("\tG              \t:",  g_diff(theta_base))
print("\tdG_dtheta      \t: ", jax.grad(g_diff)(theta_base))

Custom non differentiable solve:
	G          	: 0.8987822650513859
	dG_dtheta  	: NotImplementedError
		ERROR: This solve is non-differentiable
Custom linear solve
	G              	: 0.8987822650513859
	dG_dtheta      	:  [0.5098907  0.49111969]


So to mimic what we will be using, we will have a class A, with some data that depends on theta, and some static parameters that cannot be jitted.
It then uses an internal non-differentiable solve and implements the differentiable inverse using custom_linear_solve.

In other words:

$$ g\circ f(A^{-1}(\theta)b(\theta), \theta)$$


[one can ask why we do not use any existing database like Lineax; this is because the operator itself defines its own inverse in our case; there doesn't need to be 'solve', it is already 'solved' by a relatively simple function (the sweep). All that is left is to register that 'solve' as an custom linear solve. The only thing
that is maybe not optimal is the fact that the transpose solve is a bit cumbersome; not sure it automatically optimizes 'away' the automatic transposition... we can test both the custom VJP and custom linear solve implementation later...]

In [6]:
from dataclasses import dataclass

@jax.tree_util.register_dataclass
@dataclass(frozen=True)
class TransportOperator:
    A_mat : jnp.ndarray
    static_param : int

    @classmethod 
    def from_theta(cls, theta, static_param):
        A_mat = A_param(theta)
        return cls(A_mat = A_mat, static_param = static_param)

    def __call__(self, x):
        return self.A_mat @ x
    
    def T(self, x):
        return self.A_mat.T @ x
    
    def inv(self, b):
        return jax.lax.custom_linear_solve(matvec = lambda x : self(x),   b=b, solve = lambda _, x : self._solve(x), transpose_solve = lambda _, x : self._solve_T(x))
    def T_inv(self, b):
        return jax.lax.custom_linear_solve(matvec = lambda x : self.T(x), b=b, solve = lambda _, x : self._solve_T(x), transpose_solve = lambda _, x : self._solve(x))
    
    def _solve(self, b):                
        return _non_differentiable_solve(self.A_mat, b)
    def _solve_T(self, b):
        return _non_differentiable_solve(self.A_mat.T, b)

g_diff = lambda t : g(f(TransportOperator.from_theta(t, static_param=423).inv(b_param(t)), t), t)
print("\tG              \t:",  g_diff(theta_base))
print("\tdG_dtheta      \t: ", jax.grad(g_diff)(theta_base))


	G              	: 0.8987822650513859
	dG_dtheta      	:  [0.5098907  0.49111969]


A custom VJP version will use this like [but note it does not have forward mode differentation!]

In [7]:
from jax.extend import core
from jax.interpreters import ad


@jax.tree_util.register_dataclass
@dataclass(frozen=True)
class TransportOperatorVJP:
    A_mat : jnp.ndarray
    #static_param : int

    @classmethod 
    def from_theta(cls, theta, static_param):
        A_mat = A_param(theta)
        return cls(A_mat = A_mat)

    def __call__(self, x):
        return self.A_mat @ x
    
    def T(self, x):
        return self.A_mat.T @ x
        
    def inv(self, b):
        return inv_linear_op(self, b)    
    
    def _solve(self, b):                
        return _non_differentiable_solve(self.A_mat, b)
    def _solve_T(self, b):
        return _non_differentiable_solve(self.A_mat.T, b)

@jax.custom_vjp
def inv_linear_op(linear_op, b):
    return linear_op._solve(b)

def inv_linear_op_fwd(linear_op, b):
    x = linear_op._solve(b)
    return x, (linear_op, x, b)

def inv_linear_op_bwd(res, g):  
    linear_op, x, b = res
    psi = linear_op._solve_T(g)
    def residual_fn(params):
        linear_op, b = params
        return linear_op(x) -b    
    
    pbar = jax.vjp(residual_fn, (linear_op, b))[1](-psi)[0]
    return pbar

inv_linear_op.defvjp(inv_linear_op_fwd, inv_linear_op_bwd)
g_diff = lambda t : g(f(TransportOperatorVJP.from_theta(t, static_param=423).inv(b_param(t)), t), t)
print("\tG              \t:",  g_diff(theta_base))
print("\tdG_dtheta      \t: ", jax.grad(g_diff)(theta_base))

	G              	: 0.8987822650513859
	dG_dtheta      	:  [0.5098907  0.49111969]


####  Matrix Composition

Note that in the final implementation, we have a  $MT^{-1}$ like structure. Since we have implemented how gradients flow through the solve, composition is no issue:

In [8]:
@jax.tree_util.register_dataclass
@dataclass(frozen=True)
class MassOperator:
    mass_mat : jnp.ndarray

    @classmethod 
    def from_theta(cls, theta):
        A_mat = A_param(theta) ** theta
        return cls(A_mat)

    def __call__(self, x):
        return self.mass_mat @ x
    
    
def MTinv(mass_op, t_op, b):
    return mass_op(t_op.inv(b))

mtinv                                = lambda  t: MTinv(MassOperator.from_theta(t), TransportOperatorVJP.from_theta(t, static_param=423), b_param(t))
g_with_mass                          = lambda t : g(f(MTinv(MassOperator.from_theta(t), TransportOperatorVJP.from_theta(t, static_param=423), b_param(t)), t), t)

g_with_mass_from_custom_linear_solve = lambda t : g(f(MTinv(MassOperator.from_theta(t), TransportOperator.from_theta(t, static_param=423) , b_param(t)), t), t)

g_with_mass_naive                    = lambda t : g(f(A_param(t)**t @ jnp.linalg.solve(A_param(t) , b_param(t)), t), t)

print("Naive JAX solve")
print("\t g(MTinv(theta)):                                    ",  g_with_mass_naive(theta_base))
print("\t g(MTinv(theta)) dG_dtheta:                          ",  jax.grad(g_with_mass_naive)(theta_base))
print("Custom JVP")
print("\t g(MTinv(theta)):                                    ",  g_with_mass(theta_base))
print("\t g(MTinv(theta)) dG_dtheta:                          ",  jax.grad(g_with_mass)(theta_base))
print("Custom Linear Solve")
print("\t g(MTinv(theta)) from custom linear solve            ",  g_with_mass_from_custom_linear_solve(theta_base))
print("\t g(MTinv(theta)) dG_dtheta from custom linear solve: ",  jax.grad(g_with_mass_from_custom_linear_solve)(theta_base))


Naive JAX solve


	 g(MTinv(theta)):                                     0.6239261525094499
	 g(MTinv(theta)) dG_dtheta:                           [0.61660873 0.40572638]
Custom JVP
	 g(MTinv(theta)):                                     0.6239261525094499
	 g(MTinv(theta)) dG_dtheta:                           [0.61660873 0.40572638]
Custom Linear Solve
	 g(MTinv(theta)) from custom linear solve             0.6239261525094499
	 g(MTinv(theta)) dG_dtheta from custom linear solve:  [0.61660873 0.40572638]


## Custom transposition

We have a problem with custom linear solve: it does not allow batching, since scan assumes the result is non-linear. This is a problem; saving all the intermediate arrays can be extremely expensive, so we rather avoid it.

A simple mockup of the problem is given in the below operator:

In [88]:

@jax.tree_util.register_dataclass
@dataclass(frozen=True)
class TransportOperator:
    A_mat : jnp.ndarray
    batch_size : int

    @classmethod 
    def from_theta(cls, theta, static_param):                
        A_mat = jax.vmap(A_param, in_axes=(0,))(theta)        
        return cls(A_mat = A_mat, batch_size= static_param)

    def __call__(self, x):

        n_batches = self.A_mat.shape[0] // self.batch_size
        
        y = jnp.zeros(x.shape)
        def single_call(elem_i, x):
            return self.A_mat[elem_i] @ x[elem_i]
        
        def single_batch_call(batch_i, y):
            start = batch_i * self.batch_size            
            safe_indices = jnp.clip(jnp.arange(0, self.batch_size) + start, 0, self.A_mat.shape[0])
            return y.at[safe_indices].set(jax.vmap(single_call, in_axes=(0,None))(safe_indices, x))
        
        return jax.lax.fori_loop(0, n_batches, single_batch_call,y )
            
    def T(self, x):
        return self.A_mat.T @ x
    
    def inv(self, b):
        return jax.lax.custom_linear_solve(matvec = lambda x : self(x),   b=b, solve = lambda _, x : self._solve(x), transpose_solve = lambda _, x : self._solve_T(x))
    def T_inv(self, b):
        return jax.lax.custom_linear_solve(matvec = lambda x : self.T(x), b=b, solve = lambda _, x : self._solve_T(x), transpose_solve = lambda _, x : self._solve(x))
    
    def _solve(self, b):                
        return _solve_block_diagonal(self.A_mat, b, self.batch_size)
    def _solve_T(self, b):
        return _solve_block_diagonal(self.A_mat.T, b, self.batch_size)
    
def _solve_block_diagonal(A_mat, b, batch_size):
    n_batches = A_mat.shape[0] // batch_size

    def single_call(elem_i,b):
        return _non_differentiable_solve(A_mat[elem_i], b[elem_i])

    def single_batch_call(batch_i, y):
        start = batch_i * batch_size            
        safe_indices = jnp.clip(jnp.arange(0, batch_size) + start, 0, A_mat.shape[0])
        return y.at[safe_indices].set(jax.vmap(single_call, in_axes=(0,None))(safe_indices, b))
    
    return jax.lax.fori_loop(0, n_batches, single_batch_call,jnp.zeros(b.shape) )
    


theta_0 = jnp.linspace(0.1, 1.0, 10)
theta_1 = jnp.linspace(0.2, 1.1, 10)
theta = jnp.stack([theta_0, theta_1], axis=1)
top = TransportOperator.from_theta(theta, static_param=2)

x = jnp.linspace(0.0, 1.0, 20).reshape(10,2)

try:
    top.inv(x)
except Exception as e:
    print("ERROR: INV NOT AVAILABLE", e)



ERROR: INV NOT AVAILABLE 


In [ ]:
import lineax as lx

class TransportOperatorSolver(lx.AbstractLinearSolver):

    def __init__(self):
        pass

    def init(self,operator, options):
        return operator
    

    def compute(self, solver_state, rhs, options):
        return _solve_block_diagonal(solver_state.A_mat, rhs, solver_state.batch_size), lx.RESULTS.successful, {}
    
    def allow_dependent_columns(self, operator):
        return False
    
    def allow_dependent_rows(self, operator):
        return False
    
    def transpose(self, solver_state, options):
        return solver_state.transpose()
    
    def conj(self, solver_state, options):
        return solver_state.T()

class TransportOperator(lx.AbstractLinearOperator):
    A_mat : jnp.ndarray
    batch_size : int

    @classmethod 
    def from_theta(cls, theta, static_param):                
        A_mat = jax.vmap(A_param, in_axes=(0,))(theta)        
        return cls(A_mat = A_mat, batch_size= static_param)

    def mv(self, x):

        n_batches = self.A_mat.shape[0] // self.batch_size
        
        y = jnp.zeros(x.shape)
        def single_call(elem_i, x):
            return self.A_mat[elem_i] @ x[elem_i]
        
        def single_batch_call(batch_i, y):
            start = batch_i * self.batch_size            
            safe_indices = jnp.clip(jnp.arange(0, self.batch_size) + start, 0, self.A_mat.shape[0])
            return y.at[safe_indices].set(jax.vmap(single_call, in_axes=(0,None))(safe_indices, x))
        
        return jax.lax.fori_loop(0, n_batches, single_batch_call,y )
            
    def transpose(self, x):
        return TransportOperator(self.A_mat.T, self.batch_size)
    
    def in_structure(self):
        return jax.ShapeDtypeStruct((self.A_mat.shape[0], self.A_mat.shape[1]), self.A_mat.dtype)
    
    def out_structure(self):
        return jax.ShapeDtypeStruct((self.A_mat.shape[0], self.A_mat.shape[1]), self.A_mat.dtype)
    
    def as_matrix(self):
        assert False, "Not implemented for large operators"
        return jnp.array([[]])
    
    def is_symmetric(self):
        return False
        
    
    def inv(top, x):
        solver = TransportOperatorSolver()
        return lx.linear_solve(top, x, solver).value

    # def T_inv(self, b):
    #     return jax.lax.custom_linear_solve(matvec = lambda x : self.T(x), b=b, solve = lambda _, x : self._solve_T(x), transpose_solve = lambda _, x : self._solve(x))
    
    # def _solve(self, b):                
    #     return _solve_block_diagonal(self.A_mat, b, self.batch_size)
    # def _solve_T(self, b):
    #     return _solve_block_diagonal(self.A_mat.T, b, self.batch_size)
@lx.is_symmetric.register(TransportOperator)
def _(operator):
    return False

@lx.linearise.register(TransportOperator)
def _(operator):
    return operator


theta_0 = jnp.linspace(0.1, 1.0, 10)
theta_1 = jnp.linspace(0.2, 1.1, 10)
theta = jnp.stack([theta_0, theta_1], axis=1)
top = TransportOperator.from_theta(theta, static_param=2)

x = jnp.linspace(0.0, 1.0, 20).reshape(10,2)
y = top.inv(x)
print(top.mv(y), x)
#result = eqx.filter_jacfwd(top.inv)(top,x)

[[0.         0.05263158]
 [0.10526316 0.15789474]
 [0.21052632 0.26315789]
 [0.31578947 0.36842105]
 [0.42105263 0.47368421]
 [0.52631579 0.57894737]
 [0.63157895 0.68421053]
 [0.73684211 0.78947368]
 [0.84210526 0.89473684]
 [0.94736842 1.        ]] [[0.         0.05263158]
 [0.10526316 0.15789474]
 [0.21052632 0.26315789]
 [0.31578947 0.36842105]
 [0.42105263 0.47368421]
 [0.52631579 0.57894737]
 [0.63157895 0.68421053]
 [0.73684211 0.78947368]
 [0.84210526 0.89473684]
 [0.94736842 1.        ]]


# Actual transport operator

First, we use a single-angle transport operator:

$$ T_{gk}$$

We setup a cartesian block domain using second order tetrahedral elements

In [102]:
import jax_sn
from jax_sn.domain import CrossSectionData, Domain, TopologicalSortedAngle
from jax_sn.solution_domain import BasixLagrangianSimplex, SolutionDomain
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.mass_operator.mass_operator_base import _mass_kernel_idx_in_precomp_vmap
from jax_sn.operators.transport_operator import TransportOperatorBatched, transport_inverse_sweep_batched, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorVmapPallas
import pyvista as pv
from jax_sbgeom.jax_utils.mesh import mesh_to_pyvista_mesh
import equinox as eqx
import numpy as onp

def setup_base_case(dim, degree):
    n_elem       = 10
    n_other_dim  = 10
    other_dims   = [[(1.0, n_other_dim)]]*(dim - 1)
    xs0          = CrossSectionData.constant(0.2, 1, 1)    
    verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)]] + other_dims, [xs0], [1.0], 0)
    element_type                         = BasixLagrangianSimplex(degree, dim)
    domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections)    
    solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
    sources = jnp.repeat(sources, repeats = solution_domain.n_basis, axis = -1).reshape(-1, solution_domain.n_basis)
    return solution_domain, sources


solution_domain, sources = setup_base_case(3,2)
i_tn     = 50
angle_i  = jax_sn.quadrature_set.create_tn_quadrature_set(4)[i_tn]
toposort = TopologicalSortedAngle.from_domain_angle(solution_domain.domain, angle_i[0], 100)
x_shape  = jnp.ones((solution_domain.domain.n_elements, solution_domain.n_basis))

plotter = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh((solution_domain.domain.vertices, solution_domain.domain.connectivity)), show_edges=True)
plotter.add_arrows(onp.array([0.0,0.0,0.0]), angle_i[0], mag=2.0, color="red")
#plotter.show()
plotter.close()

Then, we parametrise the sweep by for simplicity, modifying the total cross section.

In other words, $\theta = \sigma$, and we have 

$S(theta, x) = u = T^{-1}(\theta) \circ H[x]$

Since the 'creation' of the T operator is nothing more than pointing a few arrays to some precomputed arrays, this does not have any overhead. We perform some PyTree surgery to obtain a new solution domain with the updated total cross sections and then 'create' the T operator.

Closing over the sigma parameter, we have the function that provides the above equation.

A simple objective function is just the total 'absorption' (that I pick because it depends both on $\theta$ directly and implicitly!):

$$ f(\theta, u) = \int \sigma(\theta) u dV $$

In [103]:

def get_modified_solution_domain_data(sigma, solution_domain_data):
    old_total = solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    new_total = old_total.at[0,0].set(sigma)
    return eqx.tree_at(lambda solution_domain_data : solution_domain_data.domain.cross_sections_indexed.cross_section_data.total, solution_domain_data, new_total)    


def get_transport_operator(sigma, solution_domain_data, toposort):    
    
    top = TransportOperatorBatchedPallas.from_solution_domain_data_and_toposort(get_modified_solution_domain_data(sigma, solution_domain_data), toposort, batch_size = 1000)
    return top
mass_operator   = MassOperator.from_fem_matrices_and_precomputed_geometric_data(solution_domain.fem_matrices, solution_domain.precomputed_geometric_data)
get_top_closure = partial(get_transport_operator, solution_domain_data=solution_domain.solution_domain_data, toposort=toposort)
x_base          = jnp.ones(x_shape.shape)
result_closure  = jax.jit(lambda t : get_top_closure(t).inv(mass_operator(x_base), 0))

In [11]:
%timeit result_closure(0.2).block_until_ready()

JitTracer<float64[6000,10]>


AssertionError: 

In [14]:
quad_order = 2 * solution_domain.element.element.degree
quad_points, quad_weights = solution_domain.element.volume_quadrature_order(quad_order)    
#@jax.jit
def integrate_sigma_u(solution_domain_data, solution):
    total_xs = solution_domain_data.domain.cross_sections_indexed.map_to_full_array(0)
    def _integrate_elem(elem_idx : int) -> float:
        elem_tot = jnp.full(quad_points.shape[0], elem_idx, dtype=int)        
        numerical_values  = solution_domain.element.interpolate_from_local_coordinate(quad_points, elem_tot, solution)        
        return jnp.sum(quad_weights * numerical_values )* solution_domain.precomputed_geometric_data.vol_jacobians_det[elem_idx] * total_xs.total[elem_idx]    
    return jax.vmap(_integrate_elem)(jnp.arange(solution_domain.domain.n_elements))

sigma_u_int = integrate_sigma_u(solution_domain.solution_domain_data, result_closure(0.2))
#@jax.jit
def total_closure(sigma, solution_domain_data, toposort):
    mod_solution_domain_data = get_modified_solution_domain_data(sigma, solution_domain_data)
    top                      = TransportOperatorVmapPallas(mod_solution_domain_data, toposort)#, 1000)#.inv(mass_operator(x_base), 0)
    #print(top.inv)
    
    result = top.inv(mass_operator(x_base), 0)
    return jnp.sum(integrate_sigma_u(solution_domain_data, result))

JitTracer<float64[6000,10]>


AssertionError: 

In [ ]:
#%timeit total_closure(0.2, solution_domain.solution_domain_data, toposort).block_until_ready()

In [ ]:
total_closure(0.2, solution_domain.solution_domain_data, toposort   )

Array(0.08093252, dtype=float64)

In [ ]:
try:
    jax.grad(total_closure, argnums=0)(0.2, solution_domain.solution_domain_data, toposort)
except NotImplementedError as e:
    print("NotImplementedError:", e)

NotImplementedError: JVP with aliasing not supported.


In [ ]:
q=1
a = 1
q * (a**2) * ( a - )

SyntaxError: invalid syntax (4043867554.py, line 3)

In [59]:
plot_solution = False
if plot_solution:
    result = solution_domain.interpolate(solution_domain.domain.vertices, result_closure(0.2)) ## this takes a while because the compilation of the BVH building and tracing is quite complicated...
    plotter = pv.Plotter()
    plotter.add_mesh(mesh_to_pyvista_mesh((solution_domain.domain.vertices, solution_domain.domain.connectivity)), scalars = sigma_u_int, show_edges=False)
    plotter.add_arrows(onp.array([0.0,0.0,0.0]), angle_i[0], mag=2.0, color="red")
    plotter.show()


In [37]:
plot_solution = False
if plot_solution:
    result = solution_domain.interpolate(solution_domain.domain.vertices, result_closure(0.2)) ## this takes a while because the compilation of the BVH building and tracing is quite complicated...
    plotter = pv.Plotter()
    plotter.add_mesh(mesh_to_pyvista_mesh((solution_domain.domain.vertices, solution_domain.domain.connectivity)), scalars = result, show_edges=False)
    plotter.add_arrows(onp.array([0.0,0.0,0.0]), angle_i[0], mag=2.0, color="red")
    plotter.show()
